# Assignment 2: Logistic Regression (From Scratch) - Teens Mental Health Dataset

This notebook implements **Logistic Regression from scratch** using only NumPy, without using scikit-learn's built-in classifier. We use Gradient Descent optimization with the binary cross-entropy loss function.

## Dataset
- **Teens Mental Health Dataset**
- Target: depression_label (0 = Not Depressed, 1 = Depressed)
- Features: age, gender, daily_social_media_hours, platform_usage, sleep_hours, screen_time_before_sleep, academic_performance, physical_activity, social_interaction_level, stress_level, anxiety_level, addiction_level

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

## 1. Load and Explore the Dataset

In [ ]:
# Load Teens Mental Health Dataset (Colab path)
df = pd.read_csv("/content/Teen_Mental_Health_Dataset.csv")

## 2. Data Preprocessing

### 2.1 Check for Missing Values

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

### 2.2 Handle Categorical Variables

In [ ]:
# Handle categorical variables - encode them numerically
df_encoded = df.copy()

# Label encode categorical columns
label_encoders = {}
categorical_columns = ['gender', 'platform_usage', 'social_interaction_level']

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print("Categorical encoding completed!")
for col, le in label_encoders.items():
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

### 2.3 Train-Validation-Test Split

In [ ]:
# Prepare features and target - predict depression_label
X = df_encoded.drop('depression_label', axis=1)
y = df_encoded['depression_label']

# Split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
# Split temp: 50% validation, 50% test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training samples:   {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation samples: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test samples:       {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

### 2.4 Feature Normalization

In [ ]:
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std:  {X_train_scaled.std():.6f}")

## 3. Logistic Regression Implementation (From Scratch)

### 3.1 Theory

Logistic Regression uses the sigmoid function:
- Hypothesis: h(x) = sigmoid(X * theta) = 1 / (1 + exp(-X*theta))
- Cost Function (Binary Cross-Entropy):
  J(theta) = -(1/m) * sum(y * log(h(x)) + (1-y) * log(1-h(x)))
- Gradient: grad J(theta) = (1/m) * X^T * (h(x) - y)
- Update: theta = theta - alpha * grad J(theta)

In [ ]:
class LogisticRegressionFromScratch:
    """
    Logistic Regression implemented from scratch using Gradient Descent.
    This implementation does NOT use sklearn LogisticRegression - it uses pure NumPy.
    """
    def __init__(self, learning_rate=0.1, n_epochs=1000, regularization=0.01, class_weight=None):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.regularization = regularization
        self.class_weight = class_weight
        self.theta = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def _add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]
    
    def fit(self, X, y):
        m, n = X.shape
        X_b = self._add_bias(X)
        self.theta = np.zeros(n + 1)
        
        sample_weights = np.ones(m)
        if self.class_weight is not None:
            pos_weight = self.class_weight['positive']
            neg_weight = self.class_weight['negative']
            sample_weights = np.where(y == 1, pos_weight, neg_weight)
        
        for epoch in range(self.n_epochs):
            z = X_b @ self.theta
            predictions = self._sigmoid(z)
            error = predictions - y
            weighted_error = error * sample_weights
            gradient = (1/m) * (X_b.T @ weighted_error)
            gradient[1:] += (self.regularization / m) * self.theta[1:]
            self.theta -= self.learning_rate * gradient
            predictions = np.clip(predictions, 1e-10, 1 - 1e-10)
            loss = -(1/m) * np.sum(sample_weights * (y * np.log(predictions) + (1 - y) * np.log(1 - predictions)))
            self.loss_history.append(loss)
        return self
    
    def predict_proba(self, X):
        X_b = self._add_bias(X)
        return self._sigmoid(X_b @ self.theta)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("LogisticRegressionFromScratch class defined!")

### 3.2 Handle Class Imbalance & Train the Model

In [ ]:
# Check class distribution
print("Class Distribution in Training Set:")
print(y_train.value_counts())
print(f"\nClass 0 (Not Depressed): {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"Class 1 (Depressed): {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")

# Calculate class weights to handle imbalance
# Weight = total / (2 * class_count)
n_samples = len(y_train)
n_class0 = (y_train == 0).sum()
n_class1 = (y_train == 1).sum()

class_weight = {
    'negative': n_samples / (2 * n_class0),
    'positive': n_samples / (2 * n_class1)
}

print(f"\nClass Weights:")
print(f"  Negative (0): {class_weight['negative']:.4f}")
print(f"  Positive (1): {class_weight['positive']:.4f}")

In [ ]:
# Create and train the model with class weights
log_model = LogisticRegressionFromScratch(
    learning_rate=0.1,
    n_epochs=1000,
    regularization=0.01,
    class_weight=class_weight
)

# Train
log_model.fit(X_train_scaled, y_train)

print(f"\nTraining completed!")
print(f"Final training loss (Weighted Cross-Entropy): {log_model.loss_history[-1]:.4f}")
print(f"Total epochs: {len(log_model.loss_history)}")

### 3.3 Training Loss Curve

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 6))
plt.plot(range(log_model.n_epochs), log_model.loss_history, 'b-', linewidth=1.5)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Binary Cross-Entropy Loss', fontsize=12)
plt.title('Training Loss Curve - Logistic Regression (From Scratch)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/Loss_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Initial loss: {log_model.loss_history[0]:.4f}")
print(f"Final loss: {log_model.loss_history[-1]:.4f}")
print(f"Loss reduction: {((log_model.loss_history[0] - log_model.loss_history[-1])/log_model.loss_history[0]*100):.2f}%")

### 3.4 Validation

In [ ]:
# Validate on validation set
y_val_pred = log_model.predict(X_val_scaled)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.4f}")

# Also check training accuracy for comparison
y_train_pred = log_model.predict(X_train_scaled)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy:.4f}")

if val_accuracy < train_accuracy - 0.05:
    print("\nWarning: Possible overfitting (validation much lower than training)")
else:
    print("\nNo significant overfitting detected")

### 3.5 Optimal Threshold Selection

In [ ]:
# Find optimal threshold on validation set using F1 score
y_val_prob = log_model.predict_proba(X_val_scaled)

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.1, 0.9, 0.05):
    y_pred_temp = (y_val_prob >= threshold).astype(int)
    f1_temp = f1_score(y_val, y_pred_temp)
    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Best Validation F1-Score: {best_f1:.4f}")

# Evaluate with optimal threshold on validation
y_val_pred_optimal = (y_val_prob >= best_threshold).astype(int)
val_accuracy_optimal = accuracy_score(y_val, y_val_pred_optimal)
print(f"Validation Accuracy with optimal threshold: {val_accuracy_optimal:.4f}")

## 4. Model Evaluation on Test Set

In [ ]:
# Generate predictions on test set using optimal threshold
y_pred = log_model.predict(X_test_scaled, threshold=best_threshold)
y_prob = log_model.predict_proba(X_test_scaled)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 55)
print("LOGISTIC REGRESSION MODEL EVALUATION (From Scratch)")
print("=" * 55)
print(f"Accuracy:    {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision:    {prec:.4f}")
print(f"Recall:       {rec:.4f}")
print(f"F1-Score:     {f1:.4f}")
print(f"AUC-ROC:      {auc:.4f}")
print("=" * 55)

### 4.1 Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Depressed (0)', 'Depressed (1)'],
            yticklabels=['Not Depressed (0)', 'Depressed (1)'],
            annot_kws={'size': 14})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Confusion_Matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (Not Depressed correct):  {tn}")
print(f"False Positives (Depressed wrong):       {fp}")
print(f"False Negatives (Not Depressed wrong):    {fn}")
print(f"True Positives (Depressed correct):      {tp}")

### 4.2 ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/ROC_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC = {auc:.4f}")
print("AUC of 1.0 = Perfect classifier")
print("AUC of 0.5 = Random classifier")

### 4.3 Sample Predictions with Probabilities

In [ ]:
# Sample predictions with probabilities
sample_df = pd.DataFrame({
    'Actual': y_test[:10].values,
    'Predicted': y_pred[:10],
    'Probability (Depressed)': y_prob[:10]
})
sample_df['Actual Label'] = sample_df['Actual'].map({0: 'Not Depressed', 1: 'Depressed'})
sample_df['Predicted Label'] = sample_df['Predicted'].map({0: 'Not Depressed', 1: 'Depressed'})

print("Sample Predictions with Probability Scores:")
display(sample_df[['Actual Label', 'Predicted Label', 'Probability (Depressed)']])

correct = (y_pred == y_test.values).sum()
print(f"\nCorrect predictions: {correct}/{len(y_test)} ({correct/len(y_test)*100:.1f}%)")

## 5. Summary

This notebook demonstrated:
1. **Complete Preprocessing**: Handling missing values, encoding categorical variables, train/val/test split, feature normalization
2. **From-Scratch Implementation**: Logistic Regression using pure NumPy with Gradient Descent
3. **Binary Classification**: Using sigmoid function and cross-entropy loss
4. **Comprehensive Evaluation**: Accuracy, Precision, Recall, F1-Score, AUC-ROC
5. **Visualization**: Training loss curve, confusion matrix, ROC curve

The model learns to predict depression status based on teens mental health and lifestyle factors.